---
title: "Exercise 4. MAGMA Gene Analysis"
subtitle: "Aggregate SNP associations into gene-level statistics while accounting for LD"
format:
  html:
    toc: true
    toc-depth: 3
    number-sections: true
jupyter: bash
---

# Overview

This notebook runs the MAGMA gene analysis. The gene-level test combines the SNP p-values assigned to each gene in notebook 3 while accounting for linkage disequilibrium using the 1000 Genomes European reference panel.

::: {.callout-important}
Notebook 3 must have produced `output/magma/adhd_demo.genes.annot` and optionally `output/magma/adhd_full.genes.annot` before this notebook can succeed.
:::

::: {.callout-note}
## Learning goals
By the end of this notebook, you should be able to:
- explain why LD correction is necessary in gene-based testing
- run the MAGMA gene analysis on a p-value file and annotation file
- interpret the resulting gene-level output as an aggregate signal rather than a raw SNP count
:::

::: {.callout-note}
## Questions for students

1. Why is linkage disequilibrium a problem if you simply count or average SNP-level evidence inside a gene?
2. Why does MAGMA need the study sample size when reading p-values?
3. What would happen if your LD reference ancestry was a poor match to the GWAS ancestry?
:::

In [1]:
REF_DIR=reference_data
INPUT_DIR=input/magma
OUT_DIR=output/magma
LD_REF=${REF_DIR}/g1000_eur/g1000_eur
N_TOTAL=225534

mkdir -p ${OUT_DIR}
printf 'LD reference prefix: %s\nSample size: %s\n' "${LD_REF}" "${N_TOTAL}"

LD reference prefix: reference_data/g1000_eur/g1000_eur
Sample size: 225534


# Conceptual note

Gene-based testing is not just a convenience layer. It changes the unit of inference from individual variants to genomic features that are easier to interpret biologically and more stable across LD structure.

::: {.callout-note}
MAGMA corrects for LD so that clusters of correlated SNPs do not act like independent evidence. Without that correction, some genes would look strong simply because they sit in dense LD blocks.
:::

In [4]:
run_gene_analysis() {
  local label=$1
  local annot_file=$2
  local pval_file=$3
  local out_prefix=$4

  echo
  echo '============================================================'
  echo "Running MAGMA gene analysis: ${label}"
  echo '============================================================'
  echo "Annotation file: ${annot_file}"
  echo "P-value file: ${pval_file}"
  echo "Output prefix: ${out_prefix}"

  test -f ${annot_file}
  test -f ${pval_file}

  magma \
    --bfile ${LD_REF} \
    --gene-annot ${annot_file} \
    --pval ${pval_file} use=SNP,P N=${N_TOTAL} \
    --out ${out_prefix}
}

In [5]:
run_gene_analysis FULL ${OUT_DIR}/adhd_full.genes.annot ${INPUT_DIR}/pval_full.txt ${OUT_DIR}/adhd_full


Running MAGMA gene analysis: FULL
Annotation file: output/magma/adhd_full.genes.annot
P-value file: input/magma/pval_full.txt
Output prefix: output/magma/adhd_full
Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--bfile reference_data/g1000_eur/g1000_eur
	--gene-annot output/magma/adhd_full.genes.annot
	--pval input/magma/pval_full.txt use=SNP,P N=225534
	--out output/magma/adhd_full

Start time is 14:03:57, Monday 03 Aug 2026

Loading PLINK-format data...
Reading file reference_data/g1000_eur/g1000_eur.fam... 503 individuals read
Reading file reference_data/g1000_eur/g1000_eur.bim... 22665064 SNPs read
Preparing file reference_data/g1000_eur/g1000_eur.bed... 

Reading SNP synonyms from file reference_data/g1000_eur/g1000_eur.synonyms (auto-detected)
	read 6016767 mapped synonyms from file, mapping to 3921040 SNPs in the data
	         skipped all synonym entries involved, synonymous SNPs are kept in analysis
	         writing list of detected synonyms in data to supplementary log file

In [6]:
run_gene_analysis DEMO ${OUT_DIR}/adhd_demo.genes.annot ${INPUT_DIR}/pval_demo.txt ${OUT_DIR}/adhd_demo


Running MAGMA gene analysis: DEMO
Annotation file: output/magma/adhd_demo.genes.annot
P-value file: input/magma/pval_demo.txt
Output prefix: output/magma/adhd_demo
Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--bfile reference_data/g1000_eur/g1000_eur
	--gene-annot output/magma/adhd_demo.genes.annot
	--pval input/magma/pval_demo.txt use=SNP,P N=225534
	--out output/magma/adhd_demo

Start time is 14:10:36, Monday 03 Aug 2026

Loading PLINK-format data...
Reading file reference_data/g1000_eur/g1000_eur.fam... 503 individuals read
Reading file reference_data/g1000_eur/g1000_eur.bim... 22665064 SNPs read
Preparing file reference_data/g1000_eur/g1000_eur.bed... 

Reading SNP synonyms from file reference_data/g1000_eur/g1000_eur.synonyms (auto-detected)
	read 6016767 mapped synonyms from file, mapping to 3921040 SNPs in the data
	         skipped all synonym entries involved, synonymous SNPs are kept in analysis
	         writing list of detected synonyms in data to supplementary log file

In [7]:
echo 'Gene analysis outputs:'
ls -lh ${OUT_DIR}/*.genes.out ${OUT_DIR}/*.genes.raw

Gene analysis outputs:
-rw-rw-r-- 1 samuele nogroup 5.8K Aug  3 14:12 output/magma/adhd_demo.genes.out
-rw-rw-r-- 1 samuele nogroup 5.0K Aug  3 14:12 output/magma/adhd_demo.genes.raw
-rw-rw-r-- 1 samuele nogroup 1.5M Aug  3 14:10 output/magma/adhd_full.genes.out
-rw-rw-r-- 1 samuele nogroup  11M Aug  3 14:10 output/magma/adhd_full.genes.raw


# Read the output conceptually

The `.genes.out` file gives one line per gene, including the number of SNPs used, effective parameters after LD correction, a Z-statistic, and the gene-level p-value. The `.genes.raw` file is especially important because notebook 5 uses it directly for tissue-enrichment testing.

::: {.callout-tip}
## Reflection prompts
1. Why might a gene with many assigned SNPs still fail to show significance?
2. How is the meaning of a gene-level p-value different from the p-value of the top SNP near that gene?
3. If the demo and full analyses disagree, which differences would you treat as expected and which as suspicious?
:::